In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from transformers import pipeline,Sentencr
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings 
warnings.filterwarnings("ignore")

d:\Anaconda\envs\gensim-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
import numpy as np


In [6]:
from sentence_transformers import SentenceTransformer

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
df = pd.read_csv("D:\Machine Learning\Datasets\netflix\titles.csv")
df

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5845,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021,NaN,100,"['romance', 'drama']",['NG'],NaN,tt13857480,6.8,45.0,1.466,NaN
5846,tm898842,C/O Kaadhal,MOVIE,A heart warming film that explores the concept...,2021,NaN,134,['drama'],[],NaN,tt11803618,7.7,348.0,NaN,NaN
5847,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021,NaN,90,['comedy'],['CO'],NaN,tt14585902,3.8,68.0,26.005,6.300
5848,tm1035612,Dad Stop Embarrassing Me - The Afterparty,MOVIE,"Jamie Foxx, David Alan Grier and more from the...",2021,PG-13,37,[],['US'],NaN,NaN,NaN,NaN,1.296,10.000


In [ ]:
crit = ( df['production_countries'].str.contains('IN')) & (df['type']=='MOVIE') & (df['release_year'] >= 1980)
df_ind_mov = df[crit]
df_ind_mov.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9
56,tm721687,Vaashi,MOVIE,"Ebin Mathew, a budding lawyer ambitiously join...",1983,NaN,123,"['drama', 'thriller']",['IN'],NaN,tt13913068,6.7,388.0,3.790,NaN
65,tm52815,Disco Dancer,MOVIE,"Anil, a street singer, is humiliated and drive...",1982,NaN,106,"['drama', 'romance']",['IN'],NaN,tt0208903,6.4,1476.0,7.879,5.3
66,tm159912,Agneepath,MOVIE,"The movie depicts the life of a young boy, Vij...",1990,NaN,174,"['drama', 'action', 'crime']",['IN'],NaN,tt0098999,7.6,8947.0,4.062,6.5
71,tm157603,Dil,MOVIE,Raja lives a poor lifestyle along with his dad...,1990,NaN,172,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0099429,6.6,5316.0,3.992,6.5


In [ ]:
df_ind_mov.count()

id                      570
title                   570
type                    570
description             570
release_year            570
age_certification       199
runtime                 570
genres                  570
production_countries    570
seasons                   0
imdb_id                 550
imdb_score              545
imdb_votes              545
tmdb_popularity         560
tmdb_score              545
dtype: int64

In [ ]:
df_ind_mov.loc[:, 'info'] = "Title: " + df_ind_mov['title'].copy() + " Description: " + df_ind_mov['description'].copy()
df_ind_mov.head()


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,info
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9,Title: Dostana Description: Two close friends ...
56,tm721687,Vaashi,MOVIE,"Ebin Mathew, a budding lawyer ambitiously join...",1983,NaN,123,"['drama', 'thriller']",['IN'],NaN,tt13913068,6.7,388.0,3.790,NaN,"Title: Vaashi Description: Ebin Mathew, a budd..."
65,tm52815,Disco Dancer,MOVIE,"Anil, a street singer, is humiliated and drive...",1982,NaN,106,"['drama', 'romance']",['IN'],NaN,tt0208903,6.4,1476.0,7.879,5.3,"Title: Disco Dancer Description: Anil, a stree..."
66,tm159912,Agneepath,MOVIE,"The movie depicts the life of a young boy, Vij...",1990,NaN,174,"['drama', 'action', 'crime']",['IN'],NaN,tt0098999,7.6,8947.0,4.062,6.5,Title: Agneepath Description: The movie depict...
71,tm157603,Dil,MOVIE,Raja lives a poor lifestyle along with his dad...,1990,NaN,172,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0099429,6.6,5316.0,3.992,6.5,Title: Dil Description: Raja lives a poor life...


In [ ]:
print(df_ind_mov['info'].head())

19    Title: Dostana Description: Two close friends ...
56    Title: Vaashi Description: Ebin Mathew, a budd...
65    Title: Disco Dancer Description: Anil, a stree...
66    Title: Agneepath Description: The movie depict...
71    Title: Dil Description: Raja lives a poor life...
Name: info, dtype: object


In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", token="hf_LJfVazBKDAmmoLJubhsJkMRSyaHaulEFcln")

In [ ]:
all_desc = list( df_ind_mov['info'])
out_embs = []
for r in tqdm(all_desc):
    embd= model.encode(r)
    out_embs.append(embd)
    
out_embs = np.array(out_embs)
out_embs.shape

100%|██████████| 570/570 [00:07<00:00, 78.97it/s]


(570, 384)

In [ ]:
query = 'kashmir'

def search_query ( query):
    qy_emb = model.encode([query])
    sims=[]
    for e in tqdm(out_embs):
        sims.append(model.similarity(qy_emb, e).numpy()[0])
        
    df_ind = df_ind_mov[['id', 'title', 'description']].copy()
    df_ind.loc[:, 'sims']= np.array(sims)
    return df_ind.sort_values('sims', ascending=False)[:5]

In [ ]:
def search_similar( query ):
    qy_emb = model.encode([query])
    sims=[]
    for e in tqdm(out_embs):
        sims.append(model.similarity(qy_emb, e).numpy()[0])
        
    df_ind = df_ind_mov[['id', 'title', 'description']].copy()
    df_ind.loc[:, 'sims']= np.array(sims)
    return df_ind.sort_values('sims', ascending=False)[:5]

In [ ]:
search_similar("three college friends do a lot of fun")

  0%|          | 0/570 [00:00<?, ?it/s]

100%|██████████| 570/570 [00:00<00:00, 7167.47it/s]


,id,title,description,sims
434,tm61056,Dil Chahta Hai,Three inseparable childhood friends are just o...,0.456041
721,tm182125,Zindagi Na Milegi Dobara,Three friends who were inseparable in childhoo...,0.414460
5154,tm1213009,Dear Friend,Four close friends who live together in Bangal...,0.388510
972,tm185797,Fukrey,The paths of four dream-chasing college friend...,0.387179
316,tm142564,3 Idiots,Rascal. Joker. Dreamer. Genius... You've never...,0.363208


In [ ]:
cred = pd.read_csv("C:/Users/dbda.STUDENTSDC/Desktop/AAS/Nteflix/credits.csv")
cred


,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
...,...,...,...,...,...
77796,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
77797,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
77798,373198,tm1059008,Inés Prieto,Fanny,ACTOR
77799,378132,tm1059008,Isabel Gaona,Cacica,ACTOR


In [ ]:
df_indian_mov = df[(df['production_countries'].str.contains('IN')) & (df['type']=='MOVIE') & (df['release_year'] >= 1980)]
all_ind_ids = list(df_indian_mov['id'].unique())
cred_ind = cred[cred['id'].isin(all_ind_ids)]
cred_ind.shape, cred.shape

((7209, 5), (77801, 5))

In [ ]:
# list(set(cred_ind['id']))
list(cred_ind['id'].unique())

['tm94651',
 'tm721687',
 'tm52815',
 'tm159912',
 'tm157603',
 'tm446654',
 'tm348735',
 'tm342996',
 'tm371359',
 'tm311264',
 'tm411457',
 'tm172482',
 'tm419052',
 'tm67635',
 'tm117873',
 'tm55100',
 'tm12876',
 'tm62176',
 'tm52871',
 'tm184326',
 'tm62385',
 'tm9898',
 'tm47530',
 'tm192301',
 'tm58893',
 'tm110160',
 'tm118438',
 'tm27601',
 'tm340430',
 'tm256669',
 'tm25389',
 'tm334788',
 'tm42397',
 'tm314602',
 'tm449775',
 'tm142564',
 'tm43399',
 'tm83786',
 'tm97544',
 'tm103732',
 'tm73058',
 'tm36475',
 'tm144345',
 'tm76835',
 'tm86051',
 'tm29269',
 'tm82765',
 'tm32885',
 'tm72587',
 'tm83175',
 'tm61056',
 'tm129763',
 'tm22540',
 'tm30170',
 'tm29731',
 'tm44530',
 'tm155761',
 'tm33067',
 'tm70222',
 'tm98975',
 'tm101609',
 'tm39557',
 'tm141117',
 'tm33729',
 'tm93004',
 'tm145243',
 'tm111226',
 'tm23441',
 'tm166293',
 'tm92977',
 'tm77348',
 'tm91323',
 'tm88366',
 'tm156575',
 'tm164389',
 'tm37945',
 'tm106625',
 'tm99450',
 'tm149209',
 'tm129971',
 'tm7

In [ ]:
actors = cred_ind[cred_ind['role']=='ACTOR']
dicrectors = cred_ind[cred_ind['role']=='DIRECTOR']


In [ ]:
movie_dic= []

for e in list(dicrectors['id'].unique()):
    # mov_id = list(actors[actors['id']])
    dic_ss = dicrectors[dicrectors['id']==e]
    movie_dic.append([e, ", ".join(list(dic_ss['name']))])

df_ind_dir = pd.DataFrame( movie_dic, columns=['id', 'Director'])
df_ind_dir

,id,Director
0,tm94651,Raj Khosla
1,tm721687,Vishnu Raghav
2,tm52815,Babbar Subhash
3,tm159912,Mukul Anand
4,tm157603,Indra Kumar
...,...,...
535,tm1004011,Stanley D'Costa
536,tm987730,Renuka Shahane
537,tm988613,Subhash Kapoor
538,tm1110241,Nithin Renji Panicker


In [ ]:
movie_actor = []

for e in list(actors['id'].unique()):
    # mov_id = list(actors[actors['id']])
    ss = actors[actors['id']==e]
    movie_actor.append([e, ", ".join(list(ss['name']))])

df_ind_act = pd.DataFrame( movie_actor, columns=['id', 'star cast'])
df_ind_act

,id,star cast
0,tm94651,"Amitabh Bachchan, Shatrughan Sinha, Zeenat Ama..."
1,tm721687,"Tovino Thomas, Keerthy Suresh, Kottayam Ramesh..."
2,tm52815,"Mithun Chakraborty, Kim, Rajesh Khanna, Om Shi..."
3,tm159912,"Amitabh Bachchan, Mithun Chakraborty, Madhavi,..."
4,tm157603,"Aamir Khan, Madhuri Dixit, Saeed Jaffrey, Anup..."
...,...,...
546,tm1004011,"Sooraj Pancholi, Isabelle Kaif, Sammy Jonas He..."
547,tm987730,"Kajol, Tanvi Azmi, Mithila Palkar, Kunaal Roy ..."
548,tm988613,"Richa Chadha, Saurabh Shukla, Manav Kaul, Aksh..."
549,tm1110241,"Suresh Gopi, Renji Panicker, Rajesh Sharma, Mu..."
